# 02 MLP

Self-contained MLP workflow. This notebook loads the common windows from preprocessing, extracts statistical features here, trains the MLP, evaluates it, and saves results.


## 1. MLP objective

The MLP does not use raw temporal windows. It receives one statistical feature vector per 30-second window.


## 2. Imports and configuration


In [ ]:
from pathlib import Path
import sys

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## 3. Shared training and evaluation imports


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

from wesad_utils.config import RANDOM_SEED, SEQUENCE_CHANNELS, WINDOW_SAMPLES
from wesad_utils.helpers import count_parameters, set_seed
from wesad_utils.training import (
    pos_weight_from_labels,
    save_model_artifacts,
    train_with_early_stopping,
)
from wesad_utils.evaluation import (
    binary_metrics,
    collect_probabilities,
    per_subject_metrics,
    prediction_table,
    select_threshold,
)

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
import json
import time
import joblib
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 4. Load common windows and metadata

These are the same windows used by CNN, RNN, and LSTM.


In [ ]:
sequence_dir = PROJECT_ROOT / "data" / "processed" / "sequence"
metadata_dir = PROJECT_ROOT / "data" / "processed" / "metadata"
preprocessing_dir = PROJECT_ROOT / "artifacts" / "preprocessing"

raw_paths = {
    "train": sequence_dir / "X_train_raw.pt",
    "validation": sequence_dir / "X_validation_raw.pt",
    "test": sequence_dir / "X_test_raw.pt",
}
missing_raw = [path.name for path in raw_paths.values() if not path.exists()]
if missing_raw:
    raise FileNotFoundError(
        "Raw sequence tensors are missing: "
        + ", ".join(missing_raw)
        + ". Rerun 01_preprocessing_and_splits.ipynb before this notebook."
    )

X_train_seq = torch.load(raw_paths["train"]).float()
y_train = torch.load(sequence_dir / "y_train.pt").float()
X_validation_seq = torch.load(raw_paths["validation"]).float()
y_validation = torch.load(sequence_dir / "y_validation.pt").float()
X_test_seq = torch.load(raw_paths["test"]).float()
y_test = torch.load(sequence_dir / "y_test.pt").float()

metadata_train = pd.read_csv(metadata_dir / "windows_train.csv")
metadata_validation = pd.read_csv(metadata_dir / "windows_validation.csv")
metadata_test = pd.read_csv(metadata_dir / "windows_test.csv")
with (preprocessing_dir / "sequence_channels.json").open("r", encoding="utf-8") as handle:
    saved_sequence_channels = json.load(handle)

assert saved_sequence_channels == SEQUENCE_CHANNELS
assert X_train_seq.shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
assert X_validation_seq.shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
assert X_test_seq.shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
assert torch.isfinite(X_train_seq).all()
assert torch.isfinite(X_validation_seq).all()
assert torch.isfinite(X_test_seq).all()
assert set(y_train.numpy().astype(int)).issubset({0, 1})
assert set(y_validation.numpy().astype(int)).issubset({0, 1})
assert set(y_test.numpy().astype(int)).issubset({0, 1})
assert len(metadata_train) == len(y_train)
assert len(metadata_validation) == len(y_validation)
assert len(metadata_test) == len(y_test)
assert np.array_equal(metadata_train["binary_label"].to_numpy(), y_train.numpy().astype(int))
assert np.array_equal(metadata_validation["binary_label"].to_numpy(), y_validation.numpy().astype(int))
assert np.array_equal(metadata_test["binary_label"].to_numpy(), y_test.numpy().astype(int))
assert set(metadata_train["subject_id"]).isdisjoint(set(metadata_validation["subject_id"]))
assert set(metadata_train["subject_id"]).isdisjoint(set(metadata_test["subject_id"]))
assert set(metadata_validation["subject_id"]).isdisjoint(set(metadata_test["subject_id"]))

X_train_seq.shape, X_validation_seq.shape, X_test_seq.shape


## 5. Explain why an MLP needs statistical features

Flattening `(960, 6)` raw samples would discard temporal structure poorly and create a large input. Instead, each window is summarized with statistical, BVP, EDA, temperature, and movement features.


## 6. Import shared feature-extraction functions


In [ ]:
from wesad_utils.preprocessing import extract_feature_table


## 7. Extract features from training windows


In [ ]:
X_train_features_df = extract_feature_table(X_train_seq.numpy())
X_train_features_df.shape


## 8. Extract features from validation and test windows


In [ ]:
X_validation_features_df = extract_feature_table(X_validation_seq.numpy())
X_test_features_df = extract_feature_table(X_test_seq.numpy())
X_validation_features_df.shape, X_test_features_df.shape


## 9. Fit feature scaler using training features only


In [ ]:
candidate_columns = list(X_train_features_df.columns)
exact_duplicate_mask = X_train_features_df[candidate_columns].T.duplicated()
exact_duplicate_columns = [
    column for column, is_duplicate in zip(candidate_columns, exact_duplicate_mask) if is_duplicate
]
candidate_columns = [column for column in candidate_columns if column not in exact_duplicate_columns]

correlation_matrix = X_train_features_df[candidate_columns].corr().abs()
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
high_correlation_columns = [
    column for column in upper_triangle.columns if (upper_triangle[column] > 0.95).any()
]
feature_columns = [column for column in candidate_columns if column not in high_correlation_columns]

feature_scaler = StandardScaler()
feature_scaler.fit(X_train_features_df[feature_columns])

X_train = torch.tensor(feature_scaler.transform(X_train_features_df[feature_columns]), dtype=torch.float32)
X_validation = torch.tensor(feature_scaler.transform(X_validation_features_df[feature_columns]), dtype=torch.float32)
X_test = torch.tensor(feature_scaler.transform(X_test_features_df[feature_columns]), dtype=torch.float32)

artifact_dir = PROJECT_ROOT / "artifacts" / "preprocessing"
artifact_dir.mkdir(parents=True, exist_ok=True)
with (artifact_dir / "feature_columns.json").open("w", encoding="utf-8") as handle:
    json.dump(feature_columns, handle, indent=2)
joblib.dump(feature_scaler, artifact_dir / "feature_scaler.joblib")
with (artifact_dir / "feature_selection.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "removed_exact_duplicate_columns": exact_duplicate_columns,
            "removed_high_correlation_columns": high_correlation_columns,
            "correlation_threshold": 0.95,
        },
        handle,
        indent=2,
    )

feature_dir = PROJECT_ROOT / "data" / "processed" / "features"
feature_dir.mkdir(parents=True, exist_ok=True)
X_train_features_df.to_csv(feature_dir / "X_train_unscaled.csv", index=False)
X_validation_features_df.to_csv(feature_dir / "X_validation_unscaled.csv", index=False)
X_test_features_df.to_csv(feature_dir / "X_test_unscaled.csv", index=False)
pd.DataFrame(X_train.numpy(), columns=feature_columns).to_csv(feature_dir / "X_train.csv", index=False)
pd.DataFrame(X_validation.numpy(), columns=feature_columns).to_csv(feature_dir / "X_validation.csv", index=False)
pd.DataFrame(X_test.numpy(), columns=feature_columns).to_csv(feature_dir / "X_test.csv", index=False)
X_train.shape, X_validation.shape, X_test.shape, len(exact_duplicate_columns), len(high_correlation_columns)


## 10. Create PyTorch datasets and DataLoaders


In [ ]:
train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_validation, y_validation)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 11. Define the MLP class


In [ ]:
from __future__ import annotations

import torch
from torch import nn


class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, dropout: float = 0.3) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


## 12. Display model architecture


In [ ]:
model = MLPClassifier(input_dim=X_train.shape[1]).to(device)
model


## 13. Define training loop


Training utilities above use raw logits with `BCEWithLogitsLoss`. The final model layer does not include a sigmoid.


## 14. Define validation loop


Validation collects probabilities with `torch.sigmoid(logits)` and uses validation data only to select a threshold.


## 15. Train without imbalance correction


In [ ]:
set_seed(42)
unweighted_model = MLPClassifier(input_dim=X_train.shape[1]).to(device)
unweighted_criterion = torch.nn.BCEWithLogitsLoss()
unweighted_optimizer = torch.optim.Adam(unweighted_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
unweighted_model, unweighted_history, unweighted_training_summary = train_with_early_stopping(
    unweighted_model,
    train_loader,
    validation_loader,
    unweighted_criterion,
    unweighted_optimizer,
    device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
)
unweighted_history.tail()


## 16. Train with class weights


In [ ]:
set_seed(42)
model = MLPClassifier(input_dim=X_train.shape[1]).to(device)
pos_weight = pos_weight_from_labels(y_train, device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

model, history, weighted_training_summary = train_with_early_stopping(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
)
history.tail()


## 17. Select threshold using validation data


In [ ]:
unweighted_validation_probabilities, unweighted_validation_true = collect_probabilities(unweighted_model, validation_loader, device)
unweighted_threshold, _ = select_threshold(unweighted_validation_true, unweighted_validation_probabilities)
unweighted_validation_metrics = binary_metrics(
    unweighted_validation_true,
    unweighted_validation_probabilities,
    unweighted_threshold,
)

weighted_validation_probabilities, weighted_validation_true = collect_probabilities(model, validation_loader, device)
weighted_threshold, _ = select_threshold(weighted_validation_true, weighted_validation_probabilities)
weighted_validation_metrics = binary_metrics(
    weighted_validation_true,
    weighted_validation_probabilities,
    weighted_threshold,
)

unweighted_best_epoch = int(unweighted_history.loc[unweighted_history["validation_loss"].idxmin(), "epoch"])
weighted_best_epoch = int(history.loc[history["validation_loss"].idxmin(), "epoch"])

validation_variant_comparison = pd.DataFrame(
    [
        {
            "method": "no_correction",
            "best_epoch": unweighted_best_epoch,
            "threshold": unweighted_threshold,
            "macro_f1": unweighted_validation_metrics["macro_f1"],
            "stress_recall": unweighted_validation_metrics["stress_recall"],
            "average_precision": unweighted_validation_metrics["average_precision"],
        },
        {
            "method": "class_weight",
            "best_epoch": weighted_best_epoch,
            "threshold": weighted_threshold,
            "macro_f1": weighted_validation_metrics["macro_f1"],
            "stress_recall": weighted_validation_metrics["stress_recall"],
            "average_precision": weighted_validation_metrics["average_precision"],
        },
    ]
)
display(validation_variant_comparison)

if weighted_validation_metrics["macro_f1"] >= unweighted_validation_metrics["macro_f1"]:
    selected_model = model
    selected_history = history
    selected_threshold = weighted_threshold
    selected_validation_probabilities = weighted_validation_probabilities
    selected_validation_true = weighted_validation_true
    selected_validation_metrics = weighted_validation_metrics
    selected_imbalance_method = "class_weight"
    selected_best_epoch = weighted_best_epoch
    selected_training_summary = weighted_training_summary
else:
    selected_model = unweighted_model
    selected_history = unweighted_history
    selected_threshold = unweighted_threshold
    selected_validation_probabilities = unweighted_validation_probabilities
    selected_validation_true = unweighted_validation_true
    selected_validation_metrics = unweighted_validation_metrics
    selected_imbalance_method = "no_correction"
    selected_best_epoch = unweighted_best_epoch
    selected_training_summary = unweighted_training_summary

model = selected_model
history = selected_history
threshold = selected_threshold
validation_probabilities = selected_validation_probabilities
validation_true = selected_validation_true
validation_metrics = {
    **selected_validation_metrics,
    "selected_imbalance_method": selected_imbalance_method,
    "best_validation_epoch": selected_best_epoch,
    "variant_comparison": validation_variant_comparison.to_dict(orient="records"),
}

threshold, selected_imbalance_method, validation_metrics


## 18. Evaluate test data


In [ ]:
inference_start_time = time.perf_counter()
test_probabilities, test_true = collect_probabilities(model, test_loader, device)
inference_time_seconds = time.perf_counter() - inference_start_time
test_metrics = binary_metrics(test_true, test_probabilities, threshold)
test_metrics = {**test_metrics, "inference_time_seconds": inference_time_seconds}
test_metrics


## 19. Evaluate by subject


In [ ]:
subject_metrics = per_subject_metrics(metadata_test, test_true, test_probabilities, threshold)
subject_metrics


## 20. Plot confusion matrix


In [ ]:
cm = np.array(test_metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["non-stress", "stress"])
ax.set_yticks([0, 1], labels=["non-stress", "stress"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.show()


## 21. Plot ROC and precision-recall curves


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(test_true, test_probabilities, ax=axes[0])
PrecisionRecallDisplay.from_predictions(test_true, test_probabilities, ax=axes[1])
plt.tight_layout()
plt.show()


## 22. SHAP analysis

SHAP explains model behavior, not physiological causality.


In [ ]:
try:
    import shap

    model.eval()
    shap_model = model.to("cpu")
    background = X_train[: min(100, len(X_train))]
    explain_count = min(200, len(X_validation))
    samples = X_validation[:explain_count]

    with torch.no_grad():
        print("Model output shape for SHAP:", shap_model(background[:4]).shape)

    explainer = shap.DeepExplainer(shap_model, background)
    shap_values = explainer.shap_values(samples)
    values = shap_values[0] if isinstance(shap_values, list) else shap_values
    values = np.asarray(values).squeeze()

    shap.summary_plot(values, samples.numpy(), feature_names=feature_columns, show=True)
    shap.summary_plot(values, samples.numpy(), feature_names=feature_columns, plot_type="bar", show=True)

    probs = validation_probabilities[:explain_count]
    for title, index in [("highest stress-score example", int(np.argmax(probs))), ("lowest stress-score example", int(np.argmin(probs)))]:
        explanation = shap.Explanation(
            values=values[index],
            base_values=np.asarray(explainer.expected_value).reshape(-1)[0],
            data=samples[index].numpy(),
            feature_names=feature_columns,
        )
        shap.plots.waterfall(explanation, max_display=15, show=True)

    model.to(device)
except ImportError:
    print("Install shap to run SHAP explanations: pip install shap")


## 23. Save model and results


In [ ]:
artifact_dir = PROJECT_ROOT / "artifacts" / "models" / "mlp"
test_predictions = prediction_table(metadata_test, test_true, test_probabilities, threshold)

save_model_artifacts(
    artifact_dir=artifact_dir,
    model=model,
    model_config={
        "model": "MLPClassifier",
        "input_dim": int(X_train.shape[1]),
        "dropout": 0.3,
        "parameter_count": count_parameters(model),
        "selected_imbalance_method": selected_imbalance_method,
        "best_validation_epoch": selected_best_epoch,
        "random_seed": RANDOM_SEED,
        "uses_class_weighting": selected_imbalance_method == "class_weight",
        "input_representation": "statistical_features_extracted_from_raw_windows",
        "removed_exact_duplicate_features": exact_duplicate_columns,
        "removed_high_correlation_features": high_correlation_columns,
    },
    threshold=threshold,
    history=history,
    validation_metrics=validation_metrics,
    test_metrics=test_metrics,
    per_subject=subject_metrics,
    test_predictions=test_predictions,
    training_summary={**selected_training_summary, "selected_imbalance_method": selected_imbalance_method, "inference_time_seconds": inference_time_seconds},
)
artifact_dir


## 24. Conclusion

The MLP is evaluated on the same windows and subjects as the sequence models, but it receives statistical features instead of raw temporal sequences.
